In [ ]:
import sys
sys.path.insert(0, "/home/jupyter/mlops-meta-learning/fsl-projekt/src")

In [ ]:
import numpy as np, pandas as pd, torch
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders
from fsl.models.registry import get_encoder
from fsl.models.protonet import fast_adapt

dev = torch.device("cpu")
base = dict(project="dark-data-discovery", dataset="samld", encoder="mlp",
            in_channels=26, image_size=1, n_way=5, k_shot=5, query=5, fold=0,
            log_to_vertex=False)
load_frozen, build = get_loaders(TrainConfig(**base, seed=0))
bundle, _ = load_frozen(TrainConfig(**base, seed=0))

def ev(m, tasks, n=200):
    m.eval()
    with torch.no_grad():
        a = [float(fast_adapt(m, tasks.sample(), 5, 5, 5, dev)[1]) for _ in range(n)]
    m.train()
    return float(np.mean(a))

rows = []
for seed in (0, 1, 2):
    cfg = TrainConfig(**base, seed=seed)
    tr, va, te = build(bundle, cfg)
    torch.manual_seed(seed)
    m = get_encoder(cfg)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for it in range(1, 1501):
        loss, _ = fast_adapt(m, tr.sample(), 5, 5, 5, dev)
        opt.zero_grad(); loss.backward(); opt.step()
        if it % 100 == 0:
            v, t = ev(m, va), ev(m, te)
            rows.append((seed, it, v, t))
            print(f"seed {seed} iter {it:4d} | val {v:.4f} | test {t:.4f}")

df = pd.DataFrame(rows, columns=["seed", "iter", "val", "test"])
print(df.groupby("iter")[["val", "test"]].agg(["mean", "std"]).round(4).to_string())

In [ ]:
import numpy as np, torch, torch.nn as nn
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders
from fsl.models.protonet import fast_adapt

floors = {}
for k in range(5):
    cfg = TrainConfig(project="dark-data-discovery", dataset="samld",
                      n_way=5, k_shot=5, query=5, seed=0, fold=k, log_to_vertex=False)
    load_frozen, build = get_loaders(cfg)
    bundle, _ = load_frozen(cfg)
    _, _, te = build(bundle, cfg)
    per_seed = []
    for s in (0, 1, 2):
        torch.manual_seed(s)
        m = nn.Sequential(nn.Linear(26, 64), nn.LayerNorm(64), nn.ReLU(), nn.Linear(64, 64)).eval()
        with torch.no_grad():
            a = [float(fast_adapt(m, te.sample(), 5, 5, 5, torch.device("cpu"))[1]) for _ in range(400)]
        per_seed.append(np.mean(a))
    floors[k] = float(np.mean(per_seed))
    print(f"fold {k}: podłoga testu {floors[k]:.4f} (ziarna {np.round(per_seed, 4)}) → próg {floors[k] + 0.10:.3f}")

In [ ]:
 %pip install -q pyarrow

In [ ]:
import sys, pandas, pyarrow
print(sys.executable)
print("pandas", pandas.__version__, "| pyarrow", pyarrow.__version__)
import fsl; print("fsl z:", fsl.__file__)

In [ ]:
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders

cfg = TrainConfig(project="dark-data-discovery", dataset="samld", seed=0, fold=0, log_to_vertex=False)
load_frozen, _ = get_loaders(cfg)
bundle, _ = load_frozen(cfg)

LIFT = {0: 0.114, 1: 0.1602, 2: 0.0589, 3: 0.0599, 4: 0.0957}
sus = set(bundle["schema"]["suspicious_classes"])
mark = lambda cs: ", ".join(c + ("*" if c in sus else "") for c in cs)
for f in bundle["folds"]:
    print(f"fold {f['fold']} | zysk {LIFT[f['fold']]:.3f}\n   test: {mark(f['test'])}\n   val:  {mark(f['val'])}")

In [ ]:
import numpy as np, torch
from collections import Counter
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders
from fsl.models.registry import get_encoder
from fsl.models.protonet import fast_adapt

dev = torch.device("cpu")
base_cfg = dict(project="dark-data-discovery", dataset="samld", encoder="mlp",
                in_channels=26, image_size=1, n_way=5, k_shot=5, query=5, log_to_vertex=False)
load_frozen, build = get_loaders(TrainConfig(**base_cfg, seed=0, fold=0))
bundle, _ = load_frozen(TrainConfig(**base_cfg, seed=0, fold=0))

def sample_named(s):
    """Jak GroupedEpisodeSampler.sample, ale zwraca też nazwy klas (konta rozłączne)."""
    pick = s.rng.choice(len(s.classes), s.n_way, replace=False)
    rows = []
    for ci in pick:
        info = s.by_class[s.classes[ci]]
        sa, qa = s._split_accounts(info["acct_rows"])
        sp = np.concatenate([info["acct_rows"][a] for a in sa])
        qp = np.concatenate([info["acct_rows"][a] for a in qa])
        rows += [s.rng.choice(sp, s.k_shot, replace=False), s.rng.choice(qp, s.query, replace=False)]
    return torch.from_numpy(s.X[np.concatenate(rows)]).float(), [s.classes[ci] for ci in pick]

for k in (1, 2, 3, 4):
    conf, accs = Counter(), []
    for seed in (0, 1, 2):
        cfg = TrainConfig(**base_cfg, seed=seed, fold=k)
        tr, va, _ = build(bundle, cfg)                       # test celowo pominięty
        torch.manual_seed(seed)
        m = get_encoder(cfg)
        opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        for _ in range(600):
            loss, _ = fast_adapt(m, tr.sample(), 5, 5, 5, dev)
            opt.zero_grad(); loss.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            for _ in range(400):
                x, names = sample_named(va)
                emb = m(x).reshape(5, 10, -1)
                protos = emb[:, :5].mean(1)
                pred = torch.cdist(emb[:, 5:].reshape(25, -1), protos).argmin(1).tolist()
                true = [j for j in range(5) for _ in range(5)]
                accs.append(np.mean([p == t for p, t in zip(pred, true)]))
                for p, t in zip(pred, true):
                    conf[(names[t], names[p])] += 1
    classes = sorted({a for a, _ in conf})
    print(f"\n=== fold {k} | walidacja = grupa testowa folda {(k + 1) % 5} | acc {np.mean(accs):.4f}")
    for c in classes:
        tot = sum(conf[(c, p)] for p in classes)
        print(f"  {c:22s} recall {conf[(c, c)] / tot:.3f}")
    pairs = Counter()
    for (t, p), n in conf.items():
        if t != p:
            pairs[tuple(sorted((t, p)))] += n
    errs = sum(pairs.values())
    print("  najczęstsze pomyłki:", ", ".join(f"{a}↔{b} {n / errs:.0%}" for (a, b), n in pairs.most_common(3)))

In [ ]:
import torch; print("torch przed:", torch.__version__)
%pip install -q tabpfn

In [ ]:
import numpy as np
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders

base_cfg = dict(project="dark-data-discovery", dataset="samld", encoder="mlp",
                in_channels=26, image_size=1, n_way=5, k_shot=5, query=5, log_to_vertex=False)
load_frozen, build = get_loaders(TrainConfig(**base_cfg, seed=0, fold=0))
bundle, _ = load_frozen(TrainConfig(**base_cfg, seed=0, fold=0))

def sample_named_np(s):
    pick = s.rng.choice(len(s.classes), s.n_way, replace=False)
    rows = []
    for ci in pick:
        info = s.by_class[s.classes[ci]]
        sa, qa = s._split_accounts(info["acct_rows"])
        sp = np.concatenate([info["acct_rows"][a] for a in sa])
        qp = np.concatenate([info["acct_rows"][a] for a in qa])
        rows += [s.rng.choice(sp, s.k_shot, replace=False), s.rng.choice(qp, s.query, replace=False)]
    return s.X[np.concatenate(rows)], [s.classes[ci] for ci in pick]

def nc_raw(xs, ys, xq):
    protos = np.stack([xs[ys == c].mean(0) for c in range(5)])
    return ((xq[:, None, :] - protos[None]) ** 2).sum(-1).argmin(1)

LEARNERS = {
    "nc_raw": None,
    "logreg": lambda: LogisticRegression(C=1.0, max_iter=2000),
    "1nn":    lambda: KNeighborsClassifier(n_neighbors=1),
    "rf":     lambda: RandomForestClassifier(n_estimators=200, random_state=0),
}
PROTO = {"Cycle": .562, "Fan_In": .728, "Layered_Fan_In": .399, "Bipartite": .430,
         "Stacked Bipartite": .362, "Structuring": .615, "Behavioural_Change_1": .492,
         "Behavioural_Change_2": .513, "Smurfing": .959}
WEAK = ["Cycle", "Layered_Fan_In", "Bipartite", "Stacked Bipartite",
        "Behavioural_Change_1", "Behavioural_Change_2"]

res = {n: defaultdict(lambda: [0, 0]) for n in LEARNERS}
for k in (1, 2, 3):
    _, va, _ = build(bundle, TrainConfig(**base_cfg, seed=0, fold=k))
    for _ in range(200):
        x, names = sample_named_np(va)
        x = x.reshape(5, 10, -1)
        xs, xq = x[:, :5].reshape(25, -1), x[:, 5:].reshape(25, -1)
        y = np.repeat(np.arange(5), 5)
        for n, mk in LEARNERS.items():
            pred = nc_raw(xs, y, xq) if mk is None else mk().fit(xs, y).predict(xq)
            for p, t in zip(pred, y):
                res[n][names[t]][0] += int(p == t); res[n][names[t]][1] += 1

rec = lambda n, c: res[n][c][0] / res[n][c][1]
print(f"{'klasa':22s} {'ProtoNet':>8s} " + " ".join(f"{n:>7s}" for n in LEARNERS))
for c in [c for c in PROTO if res['logreg'][c][1]]:
    flag = " *" if c in WEAK else ""
    print(f"{c:22s} {PROTO[c]:8.3f} " + " ".join(f"{rec(n, c):7.3f}" for n in LEARNERS) + flag)
print(f"\nśredni recall 6 słabych klas (*): ProtoNet 0.460 | " +
      " | ".join(f"{n} {np.mean([rec(n, c) for c in WEAK]):.3f}" for n in LEARNERS))